### 1. Selenium과 BeautifulSoup을 활용해 카테고리별 네이버 뉴스 url 크롤링

In [1]:
from selenium import webdriver

from bs4 import BeautifulSoup 
import time
import pandas as pd
import requests

import warnings
warnings.filterwarnings("ignore") # 불필요한 Warning 메시지를 꺼줍니다.

In [5]:
driver = webdriver.Chrome(executable_path='(driver) chromedriver.exe') 

In [6]:
# url 크롤링
category_num = [100, 101, 102, 103, 105]
category_name = {
    100 : '정치',
    101 : '경제',
    102 : '사회',
    103 : '생활/문화',
    105 : 'IT/과학',
}
urls_list = []
current_page = 1 # 현재 페이지
set_num = 1400   # 카테고리 별 원하는 크롤링 개수 

headers = {'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}

for i in category_num:
    print(f"\n=={category_name[i]} 카테고리 크롤링 시작")
    total_news = 0
    current_page = 1
    
    while total_news < set_num: # 카테고리 별로 원하는 크롤링 개수에 도달할 때 까지
        url = "https://news.naver.com/main/main.naver?mode=LSD&mid=shm&sid1=" + str(i) + "#&date=%2000:00:00&page=" + str(current_page)
        driver.get(url) 
        source = BeautifulSoup(driver.page_source, 'html.parser')
        
        for urls in source.find_all('a', {'class' : "sh_text_headline"}) + source.select('div.section_body a'):
            if urls["href"].startswith("https://n.news.naver.com") and urls["href"] not in urls_list and total_news < set_num:
                urls_list.append(urls["href"])
                total_news += 1
                
        current_page += 1  # 페이지 업
        print(f"현재 total_news : {total_news}")
        time.sleep(1)
    
    print(f"=={category_name[i]} 카테고리 크롤링 완료")
    time.sleep(3)


==정치 카테고리 크롤링 시작
현재 total_news : 30
현재 total_news : 30
현재 total_news : 49
현재 total_news : 68
현재 total_news : 88
현재 total_news : 108
현재 total_news : 128
현재 total_news : 148
현재 total_news : 168
현재 total_news : 188
현재 total_news : 208
현재 total_news : 228
현재 total_news : 248
현재 total_news : 268
현재 total_news : 288
현재 total_news : 308
현재 total_news : 328
현재 total_news : 348
현재 total_news : 368
현재 total_news : 388
현재 total_news : 408
현재 total_news : 428
현재 total_news : 448
현재 total_news : 468
현재 total_news : 488
현재 total_news : 508
현재 total_news : 528
현재 total_news : 548
현재 total_news : 568
현재 total_news : 588
현재 total_news : 608
현재 total_news : 628
현재 total_news : 648
현재 total_news : 668
현재 total_news : 688
현재 total_news : 708
현재 total_news : 728
현재 total_news : 748
현재 total_news : 768
현재 total_news : 788
현재 total_news : 808
현재 total_news : 828
현재 total_news : 848
현재 total_news : 868
현재 total_news : 888
현재 total_news : 908
현재 total_news : 928
현재 total_news : 948
현재 total_news : 968
현재 tota

In [7]:
driver.close()
driver.quit()

In [8]:
# 크롤링한 총 url 개수
print(f"총 url 개수 : {len(urls_list)}")

총 url 개수 : 7000


### 2. 각 url에 접근하여 세부 정보 크롤링

In [9]:
# 각 기사들의 데이터를 종류별로 나눠담을 리스트를 생성 (추후 DataFrame으로 모을 예정)
titles = []                # 제목
articles = []              # 본문
article_urls = []          # url
press_companies = []       # 발행사
categories = []            # 카테고리

#각 네이버 뉴스 url에 접근하여 크롤링 후 DataFrame 저장 
for url in urls_list:
    try:
        headers = {'User-Agent':'Mozilla/5.0 (Windows NT 6.3; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/63.0.3239.132 Safari/537.36'}
        web_news = requests.get(url, headers=headers).content
        source_news = BeautifulSoup(web_news, 'html.parser')

        title = source_news.find('h2', {'class' : 'media_end_head_headline'}).get_text()
        print('Processing article : {}'.format(title))
        
        article = source_news.find('article', {'id' : 'dic_area'}).get_text()
        article = article.replace("\n", "")
        article = article.replace("// flash 오류를 우회하기 위한 함수 추가function _flash_removeCallback() {}", "")
        article = article.replace("동영상 뉴스       ", "")
        article = article.replace("동영상 뉴스", "")
        article = article.strip()

        press_company = source_news.find('em', {'class':'media_end_linked_more_point'}).get_text()
        
        category = source_news.find('li', {'class':'Nlist_item _LNB_ITEM is_active'}).get_text().strip()
        
        titles.append(title)
        articles.append(article)
        press_companies.append(press_company)
        article_urls.append(url)
        categories.append(category)
        
        # 대량의 데이터를 대상으로 크롤링을 하기에 요청 사이에 쉬는 타이밍
        time.sleep(0.2)
    except:
        print('*** 다음 링크의 뉴스를 크롤링하는 중 에러가 발생했습니다 : {}'.format(url))


Processing article : 日, 11년째 '독도는 일본땅' 주장…"역사·국제법상 日 고유 영토"(상보)
Processing article : "김정은이 요새 대놓고 순항미사일만 쏴대는 까닭은"
Processing article : 듣도 보도 못한 TV조선의 '핼로윈 특별법'
Processing article : 이재명, 오늘 신년 회견…총선 각오 밝히고 민주당 지지 호소
Processing article : [속보]尹 대통령, '이태원특별법' 거부권 행사…취임 후 9번째
Processing article : 불교 대축전 참석한 윤석열 대통령[뉴시스Pic]
Processing article : “직장내 괴롭힘, 학폭 관련자는 얼씬도 말라”…한동훈표 공천 기준 확정, 2월말 경선 완료
Processing article : 적기 14기 격추한 美조종사 데이비스…2월의 6·25전쟁영웅
Processing article : '힘에 의한 평화' 尹대통령, 2년 연속 통합방위회의 직접 주재
Processing article : 한동훈, 오늘 수원 방문…'철도 지하화' 총선 4호 공약 발표
Processing article : 사직구장 아니라는 한동훈 "이렇게 속 좁은 정치인 처음"
Processing article : 총선 D-70, 이재명도 '명찰' 달고 면접자로... '컷오프' 예고에 떨고 있는 현역들
Processing article : 최재성 "운동권 심판론은 잘못된 총선 전략"
Processing article : "공천 신청 않는다"는 유승민 '역할론'…활용법 '의견 분분'
Processing article : 여명 "與, 10~15석 더 많아야…동대문 살릴 4대 공약 제시"[총선人]
Processing article : 선거철이 왔다…법 빈틈 노린 문자·전화 '폭탄'
Processing article : 2004년 北 룡천역 폭발, 시리아인 추적하던 모사드의 작전이었다?
Processing article : [정치쇼] 박성태 "임종석 논란? 이재명,

Processing article : 4개월짜리 의정활동 이어간다… 김근태, ‘與 탈당’ 권은희 의원직 승계
Processing article : “배현진 습격범, 범행 이유 묻자 ‘정치 이상하게 하잖아요’” 수행비서 주장
Processing article : 미니대선급 명룡대전 … "이번에도 李가 우세" "실세 元이 지역에 도움"
Processing article : 선거때마다 뜨거운 한강벨트 … 민주 "사수" 국힘 "탈환"
Processing article : [영상] 루마니아, 10번째 'K9 유저클럽' 회원 된다…운용국만 누릴 수 있는 특급혜택 뭐길래
Processing article : [팩트와이] "4호선 무임승차 최다는 경마장역"...이준석 발언은 사실일까
Processing article : [정치에 속지 않기] 尹사진 새겨진 명함이 서랍에 들어간 까닭
Processing article : [정치에 속지 않기] 尹사진 새겨진 명함이 서랍에 들어간 까닭
Processing article : 국민의힘 유종필 "험지간다며 한강 주변, 관악·강북은 사지냐"
Processing article : '오체투지'에도 이태원 특별법 거부/'독도 일본땅' 망언/양향자, 이낙연 연대 상상 안해
Processing article : [4·10 총선, 주목! 이 사람] "극한대립 종식하고 정치낭만 회복해야… 민생 바다 뛰어들것"
Processing article : 여야, 권역별 비례제로 가나…韓 “선거제 개혁 지연, 민주당 때문”
Processing article : 與 중진, 줄줄이 험지 출사표… 野 친명, 앞다퉈 비명에 도전장
Processing article : '건희사랑' 강신업 "피해자가 왜 사과하나…명품백은 던지기 수법"
Processing article : 北, 일주일 사이 순항미사일 3차례 발사…'이례적 집중' 이유는?
Processing article : 이준석이 이낙연과 통합에 소극적인 이유[여의도 앨리스]
Processing article : [정

Processing article : 尹, '도끼만행' 희생 미군 이름 딴 숙소에 헌정비…"영원히 기억"
Processing article : [4·10 총선] 여 하태경·이영·이혜훈 공천 3파전…야 친명 박성준
Processing article : 격세지감 ‘옛 권력’…공세 내몰린 친문, 부활 노리는 친박 [이런정치]
Processing article : 김경율·윤희숙 손 들어준 한동훈…정치신인·터줏대감 ‘부글부글’[이런정치]
Processing article : [정치쇼] 유상범 "尹韓 오찬에 윤재옥·이관섭도 참석…갈등 여진 정리"
Processing article : [정치쇼] 이태원 참사 유가족 "159명 압살, 그냥 덮어버리겠단 건가"
Processing article : 전여옥 “멀쩡한 차두리, 조민 남매와 엮지 마라”…‘조국부부 탄원서’ 낸 차범근 지적
Processing article : 윤건영 “친문·친명 가르는 뺄셈 정치론 尹 일방독주 못 멈춰”
Processing article : 차범근에 분개한 전여옥 "차두리랑 조민이랑 같나"
Processing article : [전격시사] 이용호 의원 (국민의힘) - “윤-한 오찬회동…결과는?”
Processing article : 조국 부부 탄원서 낸 차범근…전여옥 "차두리·조민 엮지 말라"
Processing article : 총선 앞두고 ‘3대 부조화’ 충돌…국힘·민주·신당 누구도 승리 장담 못해[Deep Read]
Processing article : 오리무중 선거제…제3지대 운명 ‘이재명 결단’에 달렸다?[이런정치]
Processing article : 고민정 "친명vs친문 프레임, 굉장히 어리석은 행동"
Processing article : ‘운동권 청산론 vs 정권 심판론’…대결 구도 윤곽 잡히나?
Processing article : 여 홍문표 "영남 공천 물갈이 대상, 그냥 물러서지 않을 것"
Processing article : [정치쇼] 윤희숙 "한동훈 사천? 얼굴도 모르는 사람을

Processing article : [취재후 Talk] 尹 대통령, '文 2019 신년 기자회견' 전철 안 밟으려면?
Processing article : 윤희숙·이혜훈·이영·하태경…與 '거물'들 서울 심장부 도전
Processing article : ‘부모가 사과했다’ 경찰 발표에 배현진측 “사과 못 받아” 반박
Processing article : '문명대전' 서막 열리나…총선 앞 '親明-親文' 갈등 고조
Processing article : ‘한강벨트 잡아라’ 與野 중량급 인사 집결한 중·성동구
Processing article : 北 “불화살-3-31형은 SLCM” 주장…핵잠 건조 둘러본 김정은 “해군 핵무장화, 절박한 시대적 과업”
Processing article : 권역별 병립형이냐, 준연동형 유지냐···정당별 선호와 유불리 평가
Processing article : [4.10 총선 `열전지대`] 野 `추미애 사단` 이성윤 전략공천설… 與 정운천, 탈환 절치부심
Processing article : 황교안, "김건희 여사 백 논란은 `몰카공작` 이전 북의 `대남공작`"
Processing article : 서천방문 6일 만에 尹 만난 한동훈 … "김건희 얘기 없었다"
Processing article : 野, 정무위서 '김건희 리스크', '이재명 피습' 공세…與는 항의, 퇴장
Processing article : 문중·토박이 영향력 막강…선거구 유지-분리 예의주시 [4·10 총선 격전지-안동예천]
Processing article : [뉴스큐] 美  "평양 방문 中 외교 부부장과 협의할 것"...무슨 얘기 하나?
Processing article : ‘86’ 노리는 여, ‘'반윤’ 모으는 야…조준점 다른 여야 총선 전략
Processing article : 北, 잠수함서 순항미사일 발사? 핵 탑재?... "기만 가능성"
Processing article : 이준석 "4호선 중 경마공원역, 무임승차 비율 최고"... 갈라치기인 이유
Processi

Processing article : 말 바꾼 이수정, ‘김건희 사과’서 “덫에 빠진 피해자”
Processing article : ‘文 저격수’ 이언주 복당? 野 홍익표 “불출마 등 선당후사 보여줘야”
Processing article : 김수철 전 부원장, 서대문구을 출마 “이재명 ‘정치 혐오’ 장본인…경의선 지하화로 서대문·마포 연결”
Processing article : 홍익표 “이언주 복당, 불출마 등 선당후사 먼저”
Processing article : "끝까지 가봐야 안다"…수도권 격전지 '승부처'
Processing article : 박지원 "김건희, 영국서 꽃가마 타고 디올백으로 세계 망신…외교가 무슨 필요?"
Processing article : [전격시사] 이수정 교수 (경기대 범죄심리학과 / 국민의힘 예비후보) - “연이은 정치인 테러…혐오정치 끝내야”
Processing article : ‘반도체 성지’ 수원 찾는 한동훈…수도권 탈환 교두보 마련할까[이런정치]
Processing article : 막말하는 극우 유튜버에 '축하 영상' 보낸 국민의힘 인사들
Processing article : 이재명 측근 “임종석 중성동갑 출마, 홍익표 선택 빛바래”
Processing article : 병장 봉급 125만원 시대…장병들 "천만원 이상 모아 여행 가겠다"
Processing article : 제3지대, 쪼그라진 ‘빅텐트론’…“이준석 바짓가랑이 잡을 이유 없다” [이런정치]
Processing article : 경형트럭 올라 강서 누빈 이준석... 與 약한고리부터 겨눈다
Processing article : 尹 '디올백' 설명…"효과 없다" vs "무조건 도움된다"
Processing article : 신평 “한동훈 與지지기반 자기 것으로…尹, 굴욕 참지 않을 것”
Processing article : 與, 경찰청장 불러 '정치인 피습' 대책 논의...여야 '총선 시계' 속도
Processing article : 한동훈, 야당에 "임종석·윤

...

Processing article : 영상 속 의상도 바꿔 준다…구글, 영상 생성 AI '루미에르' 공개[아무Tech]
Processing article : 사전판매 '121만대' 대박난 갤S24, 8년 만에 최대 판매 기대감
Processing article : "공부 잘하려면 손글씨 써야"...과학적 입증
Processing article : 예상 웃돈 실적 J&J;, 성장 지속될까[제약·바이오 해외토픽]
Processing article : 뜨거운 '단통법 폐지'…과연 '성지의 봄'이 올까요 [김준혁의 그것IT 알고싶다]
Processing article : 갤럭시 S24 얼마나 똑똑할까… 파파고·챗GPT와 비교해 봤습니다
Processing article : [PLAY IT] "잘가요 갤S10" 5년만 이별결심…갤럭시S24 울트라 환승
Processing article : 입지 줄어든 TV 등 레거시 미디어, 라이브 커머스로 눈길
Processing article : 단통법 폐지 움직임에도 갤S24 '짠물 지원금'…최대 24만원
Processing article : 美, 美클라우드 기업에 '中고객 AI 모델 학습 정보' 신고 의무화 추진
Processing article : 한국말 하면 영어로 자동번역 “이게 되네”…‘AI 스마트폰’ 성능에 깜짝 [홍키자의 빅테크]
Processing article : [겜별사] 퍼즐로 즐기는 해양 생태계 '어비스리움 매치'
Processing article : [인터뷰] 창업 17개월 만에 기업가치 1000억 돌파한 파네시아... 정명수 대표 “AI 시대 데이터 처리 증가로 ‘CXL’ 수요 늘 것”
Processing article : [오리진] 날지 못하던 이 공룡, 나는 시늉만 했는데도 먹잇감 ‘화들짝’...로봇공룡이 밝힌 ‘비밀’
Processing article : "당근서 100만원 싸게 갤S23 산다"…중고폰 시장 700만대 시대
Processing article : [김정유의 웹툰파헤치기]대악마가 본 인간

Processing article : 법원 '다크앤다커' 가처분 모두 기각…본안 소송서 다툰다
Processing article : “연구관리기관 12→4개로 통합” 주장한 박상욱 과기수석...출연연 통폐합 재점화되나
Processing article : '오픈AI' 샘 올트먼 방한…삼성·SK 경영진과 잇따라 면담
Processing article : 전세계 단 24대뿐!…순금 용 휘감은 ‘갤S24 울트라 한정판’, 가격이 무려
Processing article : 네이버·동아일보·송중기·JTBC 한꺼번에 사칭한 광고 등장
Processing article : 한국선 버티면서 유럽선 '깨갱'…애플, 앱스토어 수수료 파격 인하
Processing article : '2조 영업익' LG엔솔 "북미 사업 본격화 성과"
Processing article : 2초만에 이미지 '뚝딱'…ETRI, 초고속 생성형 시각지능기술 공개
Processing article : 제4이통 주파수 경매 2일차...출혈경쟁 될까
Processing article : "벌써부터 중도하차"…난기류 만난 제4이통 '눈치게임' 본격 개막
Processing article : 넥슨·아이언메이스 가처분 신청 기각…재판부 도용 가능성 지적
Processing article : 한국 큐브위성 달에 보내준다는 NASA의 제안, 정부가 거절했다
Processing article : [이기자의 게임픽] 찬바람 부는 게입업계, 신작-글로벌 진출로 돌파구 마련
Processing article : 인텔, 대만 파운드리 UMC와 12나노급 새 공정 공동 개발
Processing article : LGU+, 갤S24 사전예약 판매 전작보다 25% 증가
Processing article : [단통법 폐지]③정보 비대칭 여전…법제화도 까마득
Processing article : 갤럭시S24 울트라, 경도 레벨6에서도 긁었더니…'멀쩡'
Processing article : 진짜 ‘불로장생 주사’ 나올까... “늙은 쥐 젊

Processing article : HBM이 살린 SK하이닉스, 투자 2배 늘린다
Processing article : 신화 속 반인반마 닮은 미스터리 천체 켄타우로스의 비밀 [아하! 우주]
Processing article : LG전자 전장사업 2년 연속 연간 흑자…올해 성장 '드라이브' 건다
Processing article : LG전자, "올해 OLED TV 수요, 2022년 수준으로 회복 전망"
Processing article : KT, 'NFT' 접는다…블록체인 사업, 디지털 문서 중심으로 재편
Processing article : 'AI 메모리'가 살렸다, 극심한 다운턴서 탈출한 SK하이닉스
Processing article : 日 `달탐사선` 그 먼 달에 55m 오차로 착륙...비결은?
Processing article : 작은 공룡들이 사냥에 활용한 '이것'…국내 연구진이 알아냈다
Processing article : 아이티센, 말레이시아 거래소 ‘그린엑스’와 STO 협력
Processing article : LG전자 "전장 수주잔고 90조원 중반...예상치 하회"
Processing article : SM엔터, 기획사 고가인수 의혹에…카카오, 전격 감사 착수
Processing article : 카카오, SM 임원 감사 착수…"상의없이 진행한 투자 검토"
Processing article : [DD퇴근길] 제4이통 주파수 경매 경합…SK하이닉스, 흑자전환 격려금 쏜다
Processing article : 카카오 SM 경영진 PC 싹 조사했다‥본격 쇄신 작업 시동
Processing article : 野, 정부 단통법 폐지 추진에 "국민에 사과부터 해야"
Processing article : '노키아 휴대폰' 붐 이끈 이재욱 전 노키아TMC 회장 별세
Processing article : "비트코인 현물 ETF, 부자들의 놀이터 될 것...대중화 어려워" [코인브리핑]
Processing article : [이슈플러스]LG전자 “올해도 신기록

Processing article : 메타, 2년 4개월만에 시총 1조 달러 돌파
Processing article : [1분건강] 닭가슴살만 먹고 과도한 운동하면 근육 대신 통풍 생깁니다
Processing article : ‘사랑 호르몬’ 옥시토신의 새로운 효과, 자폐 치료에 새 길 열리나
Processing article : 넷플릭스, ‘프로레슬링 10년 중계권’에 6.7조원 투입
Processing article : SK하이닉스 "감산 규모 점진적 조정…보수적 투자 기조는 유지"
Processing article : [탈포털 시대 디지털 혁신] 문체 바꿔주는 AI의 등장 "기자들에게 양질의 기사 쓸 여유 주고 싶다"
Processing article : “반려견 코 사진만 찍으면 끝”…‘개민증’ 즉시 발급된다
Processing article : NASA, 유인 달 탐사에 쓸 차세대 로켓 엔진 시동 [우주로 간다]
Processing article : KT, 아시아 7개국 잇는 새로운 '해저 케이블' 건설 착수
Processing article : 고한승의 12년 결실…삼성에피스 최단기 매출 1조 달성
Processing article : '적자 탈출' SK하이닉스, 직원에 자사주·격려금 200만원 쏜다
Processing article : 삼성바이오에피스, 창립 12년 만에 매출 1조원 고지 넘었다
Processing article : 5G 28㎓ 주파수 경매 돌입…'쩐의 전쟁' 열리나
Processing article : "크롬의 제3자 쿠키 지원 중단, 온라인 광고 감소 불가피"
Processing article : 비트코인, 4만달러 회복, 그레이스케일 환매 압력 여전
Processing article : [KTVF] 파이퍼블릭 “개인도 초대형 상업용 부동산에 투자할 수 있습니다”
Processing article : “리튬메탈전지 기술 확보” LG엔솔, 美 사이온파워 지분 투자
Processing article : 애플, EU서 앱스토어 외 다

Processing article : 토종 OTT 볕도 안 드는데… 넷플릭스 제2전성기 '활짝' [IT+]
Processing article : [영상]"으악 이게 뭐야"…파리가 앉은 음식 먹어도 되나 봤더니
Processing article : 네카오, 작년 두자릿수 매출↑…영업익은 희비 갈릴 듯
Processing article : '흑자 달성' LG디스플레이, 2024년 과제는 재무건전화
Processing article : 하반신 마비 쥐, K-신약 맞고 걸었다[영상]
Processing article : [경영불안 요기요]① 이정환 대표 사임설에 '마케팅 확대' 제동 불가피
Processing article : 삼바, 영업익 첫 1조 돌파… K바이오 연대기 한획 그어
Processing article : [르포] ‘정용진 현장 경영’ 스타필드 수원, 2세대 첫 주자인 이유 있었네…역대급 터졌다


### 3. 전체 데이터 DataFrame에 모은 후 csv 파일로 저장

In [10]:
# 각 데이터 종류별 list에 담아둔 전체 데이터를 DataFrame에 모으기
article_df = pd.DataFrame({'Title':titles,  
                           'Article':articles, 
                           'URL':article_urls, 
                           'PressCompany':press_companies,
                           'Category':categories})

article_df.head()

,Title,Article,URL,PressCompany,Category
0,"日, 11년째 '독도는 일본땅' 주장…""역사·국제법상 日 고유 영토""(상보)","韓 정부 ""강력히 항의…한일 관계에 도움 안돼""가미카와 요코 일본 외무상이 지난해 ...",https://n.news.naver.com/mnews/article/421/000...,뉴스1,정치
1,"""김정은이 요새 대놓고 순항미사일만 쏴대는 까닭은""","1주일 사이 3차례나 순항미사일만 도발반길주 ""유엔 안보리 대북 제재 대상 빠진 사...",https://n.news.naver.com/mnews/article/014/000...,파이낸셜뉴스,정치
2,듣도 보도 못한 TV조선의 '핼로윈 특별법',이태원 참사 특별법을 '핼로윈 특별법'이라 칭해... 정식 명칭과도 안 맞아▲ 3...,https://n.news.naver.com/mnews/article/047/000...,오마이뉴스,정치
3,"이재명, 오늘 신년 회견…총선 각오 밝히고 민주당 지지 호소",尹정권 비판하며 대안 제시…선거제 입장도 주목발언하는 이재명 대표(서울=연합뉴스) ...,https://n.news.naver.com/mnews/article/001/001...,연합뉴스,정치
4,"[속보]尹 대통령, '이태원특별법' 거부권 행사…취임 후 9번째",윤석열 대통령이 30일 이태원특별법(10·29 이태원참사 피해자 권리보장과 진상규명...,https://n.news.naver.com/mnews/article/658/000...,국제신문,정치


In [11]:
len(article_df)

7000

In [ ]:
# 크롤링 정보를 csv 파일로 저장
article_df.to_csv('dataset/naver_news_category.csv', index=False)